#### Imports and load

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2

# Load the data
train = pd.read_csv('../data/raw/fraudTrain.csv')
test = pd.read_csv('../data/raw/fraudTest.csv')

# Drop junk column
train = train.drop(columns=['Unnamed: 0'])
test = test.drop(columns=['Unnamed: 0'])

print("Data loaded successfully")
print(f"Train: {train.shape} | Test: {test.shape}")

Data loaded successfully
Train: (1296675, 22) | Test: (555719, 22)


In [4]:
print(train['trans_date_trans_time'])

0          2019-01-01 00:00:18
1          2019-01-01 00:00:44
2          2019-01-01 00:00:51
3          2019-01-01 00:01:16
4          2019-01-01 00:03:06
                  ...         
1296670    2020-06-21 12:12:08
1296671    2020-06-21 12:12:19
1296672    2020-06-21 12:12:32
1296673    2020-06-21 12:13:36
1296674    2020-06-21 12:13:37
Name: trans_date_trans_time, Length: 1296675, dtype: str


#### Time Features

In [5]:
# Convert transaction time from string to datetime object
train['trans_date_trans_time'] = pd.to_datetime(train['trans_date_trans_time'])
test['trans_date_trans_time'] = pd.to_datetime(test['trans_date_trans_time'])

# Extract meaningful time components from the training set
train['hour'] = train['trans_date_trans_time'].dt.hour
train['day_of_week'] = train['trans_date_trans_time'].dt.day_of_week
train['month'] = train['trans_date_trans_time'].dt.month
train['is_weekend'] = train['day_of_week'].isin([5, 6]).astype(int)

# Extract meaningful time components from the test set
test['hour'] = test['trans_date_trans_time'].dt.hour
test['day_of_week'] = test['trans_date_trans_time'].dt.day_of_week
test['month'] = test['trans_date_trans_time'].dt.month
test['is_weekend'] = test['day_of_week'].isin([5,6]).astype(int)

# Quick check - fraud by hour
fraud_by_hour = train.groupby('hour')['is_fraud'].mean() * 100
print("Fraud rate by hour:")
print(fraud_by_hour.round(2))

Fraud rate by hour:
hour
0     1.49
1     1.53
2     1.47
3     1.42
4     0.11
5     0.14
6     0.09
7     0.13
8     0.12
9     0.11
10    0.09
11    0.10
12    0.10
13    0.12
14    0.13
15    0.12
16    0.12
17    0.12
18    0.12
19    0.12
20    0.10
21    0.11
22    2.88
23    2.84
Name: is_fraud, dtype: float64


#### Age Feature

In [6]:
# Convert date of birth to datetime
train['dob'] = pd.to_datetime(train['dob'])
test['dob'] = pd.to_datetime(test['dob'])

# Calculate age at time of transaction
train['age'] = (train['trans_date_trans_time'] - train['dob']).dt.days // 365
test['age'] = (test['trans_date_trans_time'] - test['dob']).dt.days // 365

# Check age distribution by fraud class
print(train.groupby('is_fraud')['age'].describe())

              count       mean        std   min   25%   50%   75%   max
is_fraud                                                               
0         1289169.0  45.511960  17.398818  13.0  32.0  43.0  57.0  95.0
1            7506.0  48.321609  18.864543  14.0  33.0  47.0  60.0  93.0


#### Distance Feature

In [7]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculates the distance in km between 2 geographic coordinates.
    Uses the Haversine formula and accounts for Earth's curvature.
    """

    R = 6371 # Earth's radius in km

    # Convert degrees to radians for the haversine formula
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    # Diff in coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c
# Apply to every row
train['distance_km'] = train.apply(
    lambda row: haversine_distance(
        row['lat'], row['long'],
        row['merch_lat'], row['merch_long']
    ), axis=1
)

test['distance_km'] = test.apply(
    lambda row: haversine_distance(
        row['lat'], row['long'],
        row['merch_lat'], row['merch_long']
    ), axis=1
)

# Compare distance for fraud vs legitimate
print(train.groupby('is_fraud')['distance_km'].describe())


              count       mean        std       min        25%        50%  \
is_fraud                                                                    
0         1289169.0  76.113756  29.119051  0.022255  55.332701  78.233012   
1            7506.0  76.268330  28.752602  0.738769  55.632890  77.931954   

                75%         max  
is_fraud                         
0         98.504498  152.117173  
1         98.391090  144.522410  


#### Behavioural Features

In [8]:
# For each card, calculate their historial spending patterns
card_stats = train.groupby('cc_num').agg(
    avg_amount = ('amt', 'mean'),
    std_amount = ('amt', 'std'),
    total_transactions = ('amt', 'count'),
    max_amount = ('amt', 'max')
).reset_index()

# Merge back onto train
train = train.merge(card_stats, on='cc_num', how='left')
test = test.merge(card_stats, on='cc_num', how='left')

# How unsual is this transaction relative to the card's history
train['amt_deviation'] = (train['amt'] - train['avg_amount']) / (train['std_amount'] + 1)
test['amt_deviation'] = (test['amt'] - test['avg_amount']) / (test['std_amount'] + 1)

print(train.groupby('is_fraud')['amt_deviation'].describe())


              count      mean       std       min       25%       50%  \
is_fraud                                                                
0         1289169.0 -0.018499  0.931084 -1.071495 -0.392483 -0.179659   
1            7506.0  3.177301  3.289981 -2.812609  0.067994  2.235358   

               75%        max  
is_fraud                       
0         0.088732  52.733393  
1         6.030437  15.576116  


#### Vectorisation of the haversine function

In [9]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    """
    Same formula but operates on entire columns at once instead of row by row
    """

    R = 6371

    # np.radians converts the entire column at once
    # instead of one value at a time

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return R * c

# This processes all 1.3 million rows simultaneously
train['distance_km'] = haversine_vectorized(
    train['lat'], train['long'],
    train['merch_lat'], train['merch_long']
)

#### Feature Engineering Decision — Cold Start Handling

Cardholders with fewer than 5 transactions do not have a stable 
spending baseline. For these cardholders, amt_deviation is set to 0 
(neutral) rather than computed, to avoid inflated deviation scores 
caused by insufficient history.

This is a known challenge in fraud detection called the cold start 
problem. In production, alternative signals would be used for new 
cardholders, such as device fingerprinting, IP geolocation, and 
merchant risk scores.

In [10]:
MIN_TRANSACTIONS = 5 # defining what "enough history" means

train['amt_deviation'] = np.where(
    train['total_transactions'] >= MIN_TRANSACTIONS,
    (train['amt'] - train['avg_amount']) / (train['std_amount'] + 1),
    0 # for new cardholders, deviation is 0 = neutral
)

test['amt_deviation'] = np.where(
    test['total_transactions'] >= MIN_TRANSACTIONS,
    (test['amt'] - test['avg_amount']) / (test['std_amount'] + 1),
    0
)